# ***Libraries***

In [ ]:
import pandas as pd
import re
import nltk
import string
import matplotlib.pyplot as plt
import seaborn as sns
import numpy as np

from nltk.tokenize import word_tokenize
from nltk.corpus import stopwords
from nltk.stem import WordNetLemmatizer

from sklearn.model_selection import train_test_split
from sklearn.feature_extraction.text import CountVectorizer, TfidfVectorizer
from sklearn.naive_bayes import MultinomialNB
from sklearn.svm import LinearSVC
from sklearn.metrics import accuracy_score, classification_report,confusion_matrix
import warnings
warnings.filterwarnings('ignore')

from transformers import BertTokenizer, BertForSequenceClassification
import torch
from transformers import Trainer, TrainingArguments

In [ ]:
nltk.download('punkt')
nltk.download('stopwords')
nltk.download('wordnet')
nltk.download('punkt_tab')

In [ ]:
df = pd.read_csv("/content/Suicide_Detection.csv",
                 usecols=[0, 1, 2],
                 encoding='latin1')

In [ ]:
df.head()

In [ ]:
df.columns = ['id', 'text', 'class']
df.head()

In [ ]:
df.info()

In [ ]:
df.isnull().sum()

In [ ]:
df = df.drop(columns=['id'])

In [ ]:
df.dropna(subset=['text', 'class'], inplace=True)
valid_classes = ['suicide', 'non-suicide']
df = df[df['class'].isin(valid_classes)]

In [ ]:
df.head()

In [ ]:
df.isnull().sum()

In [ ]:
print("Data Info after cleaning:")
df.info()

In [ ]:
df['class'].value_counts()

In [ ]:
sns.countplot(x='class',data=df)
plt.title("suicide vs non-suicide distribution")
plt.show()

In [ ]:
df['class'] = df['class'].map({'suicide': 1, 'non-suicide': 0})

In [ ]:
df.head()

In [ ]:
def clean_text(text):
    text = text.lower()
    text = re.sub(r'http\S+|www\S+|https\S+', '', text, flags=re.MULTILINE)
    text = re.sub(r'[^\x00-\x7f]',r' ', text)
    text = re.sub(r'[^a-zA-Z\s]', '', text)
    text = re.sub(r'\s+', ' ', text).strip()
    return text

df['clean_text'] = df['text'].apply(clean_text)

df[['text', 'clean_text']].head()

In [ ]:
df['tokens'] = df['clean_text'].apply(word_tokenize)

In [ ]:
stop_words = set(stopwords.words('english'))
lemmatizer = WordNetLemmatizer()

def preprocess_text(tokens):
  words = [lemmatizer.lemmatize(word) for word in tokens if word not in stop_words]
  return ' '.join(words)

df['Clean_Text'] = df['tokens'].apply(preprocess_text)

In [ ]:
df = df[df['Clean_Text'].str.strip() != ""]

In [ ]:
output = df[['Clean_Text', 'class']]
output.to_csv('MindWatch_Cleaned.csv', index=False)

print('Saved: MindWatch_Cleaned.csv')
print('Shape:', output.shape)
output.head()


1. **Vectorization** — Bag-of-Words & TF-IDF (unigram and bigram)
2. **Train / Test Split** — Stratified 80/20
3. **Model Training** — Naive Bayes & SVM
4. **N-gram Comparison** — Results table across all configurations

### 1. Train / Test Split

In [ ]:
X = df['Clean_Text'].values
y = df['class'].values

X_train, X_test, y_train, y_test = train_test_split(
    X, y,
    test_size=0.20,
    random_state=42,
    stratify=y          # keeps suicide/non-suicide ratio equal in both sets
)

print(f'Training samples : {len(X_train):,}')
print(f'Testing  samples : {len(X_test):,}')

### 2. Vectorization
We build **four** vectorizers to compare

| Config | Type | N-gram range |
|---|---|---|
| `BoW_uni` | CountVectorizer | (1,1) |
| `BoW_bi` | CountVectorizer | (1,2) |
| `TFIDF_uni` | TfidfVectorizer | (1,1) |
| `TFIDF_bi` | TfidfVectorizer | (1,2) |

In [ ]:
COMMON = dict(strip_accents='unicode', min_df=2, max_df=0.95)

vectorizers = {
    'BoW_uni'   : CountVectorizer(ngram_range=(1, 1), **COMMON),
    'BoW_bi'    : CountVectorizer(ngram_range=(1, 2), **COMMON),
    'TFIDF_uni' : TfidfVectorizer(ngram_range=(1, 1), **COMMON),
    'TFIDF_bi'  : TfidfVectorizer(ngram_range=(1, 2), **COMMON),
}

# Fit ONLY on train → transform both (prevents data leakage)
X_train_vec, X_test_vec = {}, {}
for name, vec in vectorizers.items():
    X_train_vec[name] = vec.fit_transform(X_train)
    X_test_vec[name]  = vec.transform(X_test)
    print(f'{name:12s}  vocab: {len(vec.vocabulary_):,}  '
          f'train shape: {X_train_vec[name].shape}')

### 3. Model Training

In [ ]:
def train_evaluate(model, X_tr, X_te, y_tr, y_te, label):
    model.fit(X_tr, y_tr)
    y_pred = model.predict(X_te)
    return {'label': label,
            'accuracy': round(accuracy_score(y_te, y_pred), 4),
            'y_pred': y_pred,
            'model': model}

nb_results, svm_results = {}, {}

print('=== Naive Bayes ===')
for vname in vectorizers:
    r = train_evaluate(MultinomialNB(alpha=1.0),
                       X_train_vec[vname], X_test_vec[vname],
                       y_train, y_test, f'NB + {vname}')
    nb_results[vname] = r
    print(f"  {r['label']:22s}  accuracy = {r['accuracy']:.4f}")

print()
print('=== SVM (LinearSVC) ===')
for vname in vectorizers:
    r = train_evaluate(LinearSVC(C=1.0, max_iter=2000, random_state=42),
                       X_train_vec[vname], X_test_vec[vname],
                       y_train, y_test, f'SVM + {vname}')
    svm_results[vname] = r
    print(f"  {r['label']:23s}  accuracy = {r['accuracy']:.4f}")

### 4. N-gram Comparison

In [ ]:
import pandas as pd

rows = []
for vname in vectorizers:
    nb_acc  = nb_results[vname]['accuracy']
    svm_acc = svm_results[vname]['accuracy']
    rows.append({'Vectorizer': vname,
                 'Naive Bayes': nb_acc,
                 'SVM': svm_acc,
                 'Best': max(nb_acc, svm_acc)})

comp = pd.DataFrame(rows).set_index('Vectorizer')
print(comp.to_string())
best_vec = comp['Best'].idxmax()
print(f'\n✓ Best config: {best_vec}  (accuracy = {comp.loc[best_vec, "Best"]:.4f})')

### 5. Detailed Classification Report (Best Configuration)

In [ ]:
best_vec = comp['Best'].idxmax()
use_svm  = svm_results[best_vec]['accuracy'] >= nb_results[best_vec]['accuracy']
best_res = svm_results[best_vec] if use_svm else nb_results[best_vec]

print(f"Best model: {best_res['label']}\n")
print(classification_report(y_test, best_res['y_pred'],
                            target_names=['non-suicide', 'suicide']))

# **BERT**

In [ ]:
!pip install transformers datasets torch

In [ ]:
df_sample = df.groupby('class').sample(n=5000, random_state=42)

train_texts, val_texts, train_labels, val_labels = train_test_split(
    df_sample['clean_text'].tolist(),
    df_sample['class'].tolist(),
    test_size=0.2,
    random_state=42
)

In [ ]:
tokenizer = BertTokenizer.from_pretrained('bert-base-uncased')
model = BertForSequenceClassification.from_pretrained('bert-base-uncased', num_labels=2)

train_encodings = tokenizer(train_texts, truncation=True, padding=True, max_length=128)
val_encodings = tokenizer(val_texts, truncation=True, padding=True, max_length=128)


In [ ]:
class MindWatchDataset(torch.utils.data.Dataset):
    def __init__(self, encodings, labels):
        self.encodings = encodings
        self.labels = labels

    def __getitem__(self, idx):
        item = {key: torch.tensor(val[idx]) for key, val in self.encodings.items()}
        item['labels'] = torch.tensor(self.labels[idx])
        return item

    def __len__(self):
        return len(self.labels)

train_dataset = MindWatchDataset(train_encodings, train_labels)
val_dataset = MindWatchDataset(val_encodings, val_labels)

In [ ]:
training_args = TrainingArguments(
    output_dir='./results',
    num_train_epochs=3,
    per_device_train_batch_size=16,
    per_device_eval_batch_size=16,
    warmup_steps=500,
    weight_decay=0.01,
    logging_dir='./logs',
    logging_steps=10,
    eval_strategy="epoch",
    save_strategy="epoch",
    load_best_model_at_end=True,
    report_to="none"
)

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=train_dataset,
    eval_dataset=val_dataset
)

print("Starting Fine-tuning... Please wait.")
trainer.train()

print("BERT Training Completed Successfully!")

In [ ]:
predictions = trainer.predict(val_dataset)
y_pred = np.argmax(predictions.predictions, axis=1)
y_true = val_labels

print(f"Overall Accuracy: {accuracy_score(y_true, y_pred)*100:.2f}%")
print("\nDetailed Report:")
print(classification_report(y_true, y_pred, target_names=['Non-Suicide', 'Suicide']))

In [ ]:
plt.figure(figsize=(8,6))
cm = confusion_matrix(y_true, y_pred)
sns.heatmap(cm, annot=True, fmt='d', cmap='pink',
            xticklabels=['Non-Suicide', 'Suicide'],
            yticklabels=['Non-Suicide', 'Suicide'])
plt.xlabel('Predicted Label')
plt.ylabel('True Label')
plt.title('Confusion Matrix - MindWatch BERT Model')
plt.show()

In [ ]:
from sklearn.manifold import TSNE
import torch

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
model.to(device)

sample_size = 500
sample_encodings = tokenizer(val_texts[:sample_size], truncation=True, padding=True, max_length=128, return_tensors="pt").to(device)

with torch.no_grad():
    outputs = model.bert(**sample_encodings)
    embeddings = outputs.last_hidden_state[:, 0, :].cpu().numpy()

tsne = TSNE(n_components=2, random_state=42)
low_dim_embeddings = tsne.fit_transform(embeddings)

plt.figure(figsize=(10, 8))
labels = np.array(val_labels[:sample_size])
colors = ['#1f77b4', '#ff7f0e']
for i, label in enumerate(['Non-Suicide', 'Suicide']):
    indices = np.where(labels == i)
    plt.scatter(low_dim_embeddings[indices, 0], low_dim_embeddings[indices, 1], label=label, alpha=0.7, c=colors[i], edgecolors='w')

plt.legend()
plt.title('MindWatch - BERT Semantic Space Visualization')
plt.grid(True, linestyle='--', alpha=0.3)
plt.show()

In [ ]:
import joblib

best_vectorizer = vectorizers[best_vec]
best_ml_model = best_res['model']

joblib.dump(best_vectorizer, 'best_vectorizer.pkl')
joblib.dump(best_ml_model, 'best_ml_model.pkl')
print("Saved traditional pipeline components successfully!")


model.save_pretrained('./saved_bert_model')
tokenizer.save_pretrained('./saved_bert_model')
print("Saved BERT model components successfully!")

In [ ]:
!pip install streamlit -q
!npm install -g localtunnel -q

In [ ]:
%%writefile app.py
import streamlit as st
import torch
import joblib
import re
import numpy as np
from transformers import BertTokenizer, BertForSequenceClassification

# Page Configuration
st.set_page_config(page_title="MindWatch Platform", page_icon="🧠", layout="centered")

# Cached function to load models once
@st.cache_resource
def load_models():
    vectorizer = joblib.load('best_vectorizer.pkl')
    ml_model = joblib.load('best_ml_model.pkl')
    bert_tokenizer = BertTokenizer.from_pretrained('./saved_bert_model')
    bert_model = BertForSequenceClassification.from_pretrained('./saved_bert_model')
    bert_model.eval()
    return vectorizer, ml_model, bert_tokenizer, bert_model

try:
    vectorizer, ml_model, bert_tokenizer, bert_model = load_models()
except Exception as e:
    st.error("Error: Make sure you have executed the training code and saved the models first!")

# Text Cleaning Function
def clean_text(text):
    text = text.lower()
    text = re.sub(r'http\S+|www\S+|https\S+', '', text, flags=re.MULTILINE)
    text = re.sub(r'[^\x00-\x7f]', r' ', text)
    text = re.sub(r'[^a-zA-Z\s]', '', text)
    text = re.sub(r'\s+', ' ', text).strip()
    return text

# UI Layout
st.title("🧠 MindWatch: Suicide Risk Detection Platform")
st.markdown("Analyze online posts, essays, or text sequences to detect potential indications of self-harm or suicide risks.")
st.markdown("---")

# Input Components
user_input = st.text_area("Input Text Content:", height=150, placeholder="Type or paste your text here...")
model_option = st.selectbox("Select Model Core:", ("Fine-tuned BERT Transformer", "Traditional ML Model (SVM / Naive Bayes)"))

# Inference Trigger
if st.button("Run MindWatch Analysis"):
    if user_input.strip() == "":
        st.warning("Please provide a valid text input before analyzing.")
    else:
        cleaned_text = clean_text(user_input)

        with st.spinner("Analyzing semantics and context..."):
            if model_option == "Traditional ML Model (SVM / Naive Bayes)":
                vec_text = vectorizer.transform([cleaned_text])
                prediction = ml_model.predict(vec_text)[0]
                confidence_info = "Classification based on the model's decision boundary."
            else:
                inputs = bert_tokenizer(cleaned_text, truncation=True, padding=True, max_length=128, return_tensors="pt")
                with torch.no_grad():
                    outputs = bert_model(**inputs)
                    probs = torch.nn.functional.softmax(outputs.logits, dim=-1).flatten().tolist()
                    prediction = np.argmax(probs)
                    confidence_info = f"Transformer Intent Confidence: {probs[prediction]*100:.2f}%"

        # Results Presentation
        st.subheader("Analysis Breakdown:")

        if prediction == 1:
            st.error("Flagged Notification: Suicide Intent Risk Detected.")
            st.metric(label="Status", value="High Risk Indicator")
        else:
            st.success("Analysis Clear: Non-Suicide Text Profile.")
            st.metric(label="Status", value="Low/No Risk Detected")

        st.info(f"ℹ**Metadata Details:** {confidence_info}")

In [ ]:
!curl ipv4.icanhazip.com

In [ ]:
!streamlit run app.py & npx localtunnel --port 8501

In [ ]:
!pip install pyngrok -q

In [ ]:
from pyngrok import ngrok

NGROK_AUTH_TOKEN = "3EXMpviuYX6SaNy9hOItoxVE9Oc_2YbDoZMGd8FvW6rs4RVcD"
ngrok.set_auth_token(NGROK_AUTH_TOKEN)

import os
os.system("streamlit run app.py --server.port 8501 &")

public_url = ngrok.connect(8501, proto="http")
print("Your App URL is:", public_url)